# Ålands bussnät – GTFS-utforskning och linjeval

Det här verktyget läser in **GTFS** (hållplatser, linjer och tidtabell), visar **hela nätet på en interaktiv
karta** och låter dig **välja ut de linjer** du vill titta närmare på / optimera senare.

**Arbetsgång**
1. Välj mappen (eller `.zip`) med GTFS-filerna.
2. Se hela nätet på kartan. **Klicka på en linje** för att se antal avgångar per dag, avgångar per timme i
   förmiddags- (06–09) och eftermiddagsrusning (15–18), samt **riktning** (går linjen åt båda hållen eller bara ett).
3. Välj vilka linjer som ska ingå i det fortsatta arbetet (en avgränsning – vi tittar bara på en delmängd av nätet).

> Optimeringen (genetiska algoritmen) ligger på is tills vidare – detta verktyg handlar om att förstå och avgränsa nätet.

## (Valfritt) Google Colab
Kör cellen om du är i Colab och vill montera din Google Drive (så du kan peka `GTFS_PATH` mot en mapp där, och
spara kartorna). Utanför Colab gör den ingenting.

In [ ]:
import sys, os
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive monterad. Lägg din GTFS-mapp/zip någonstans under /content/drive/MyDrive/ och ange sökvägen nedan.')
else:
    print('Inte i Colab – kör lokalt som vanligt.')

## Beroenden
`pip install pandas numpy folium matplotlib ipywidgets`

In [ ]:
import os, math, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import folium
    from folium.plugins import PolyLineTextPath
    HAR_FOLIUM = True
except Exception:
    HAR_FOLIUM = False
    print('Folium saknas – kartan ritas statiskt. Installera med: pip install folium')

## GTFS-funktioner
Inläsning, statistik per linje samt kartor. (Behöver normalt inte ändras.)

In [ ]:
GTFS_FILER = ['agency', 'stops', 'routes', 'trips', 'stop_times',
              'calendar', 'calendar_dates', 'shapes', 'frequencies']

def las_gtfs(path):
    """Läser en GTFS-mapp ELLER .zip till en dict av DataFrames (allt som text)."""
    tab = {}
    if str(path).lower().endswith('.zip'):
        with zipfile.ZipFile(path) as zf:
            namn = {os.path.basename(n).replace('.txt', ''): n
                    for n in zf.namelist() if n.endswith('.txt')}
            for t in GTFS_FILER:
                if t in namn:
                    with zf.open(namn[t]) as fh:
                        tab[t] = pd.read_csv(fh, dtype=str, keep_default_na=False)
    else:
        for t in GTFS_FILER:
            fp = os.path.join(path, t + '.txt')
            if os.path.exists(fp):
                tab[t] = pd.read_csv(fp, dtype=str, keep_default_na=False)
    if 'stops' not in tab or 'routes' not in tab or 'trips' not in tab or 'stop_times' not in tab:
        raise FileNotFoundError('GTFS ofullständig: stops/routes/trips/stop_times krävs. Hittade: '
                                + ', '.join(sorted(tab)))
    # numeriska kolumner
    for c in ['stop_lat', 'stop_lon']:
        tab['stops'][c] = pd.to_numeric(tab['stops'][c], errors='coerce')
    tab['stop_times']['stop_sequence'] = pd.to_numeric(tab['stop_times']['stop_sequence'], errors='coerce')
    if 'direction_id' not in tab['trips'].columns:
        tab['trips']['direction_id'] = '0'
    tab['trips']['direction_id'] = tab['trips']['direction_id'].replace('', '0')
    return tab

def gtfs_tid_min(s):
    """GTFS-tid 'HH:MM:SS' (kan vara >24h) -> minuter efter midnatt (float)."""
    if not s or ':' not in str(s):
        return np.nan
    d = str(s).split(':')
    return int(d[0])*60 + int(d[1]) + (int(d[2])/60 if len(d) > 2 else 0)

# ------------------------------------------------------------------ aktiva turer en viss dag
def aktiva_services(gtfs, datum):
    """service_id:n som trafikeras ett visst datum (datetime.date). Hanterar calendar + calendar_dates."""
    aktiva = set()
    vd = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday'][datum.weekday()]
    dstr = datum.strftime('%Y%m%d')
    if 'calendar' in gtfs:
        cal = gtfs['calendar']
        for _, r in cal.iterrows():
            if r.get(vd, '0') == '1' and r.get('start_date', '00000000') <= dstr <= r.get('end_date', '99999999'):
                aktiva.add(r['service_id'])
    if 'calendar_dates' in gtfs:
        for _, r in gtfs['calendar_dates'].iterrows():
            if r['date'] == dstr:
                if r['exception_type'] == '1':
                    aktiva.add(r['service_id'])
                elif r['exception_type'] == '2':
                    aktiva.discard(r['service_id'])
    return aktiva

def valj_standarddatum(gtfs):
    """Föreslår ett representativt vardagsdatum som faktiskt har trafik."""
    idag = datetime.date.today()
    for off in range(0, 21):
        d = idag + datetime.timedelta(days=off)
        if d.weekday() < 5 and aktiva_services(gtfs, d):
            return d
    # annars: ta första giltiga datum i calendar
    if 'calendar' in gtfs and len(gtfs['calendar']):
        s = gtfs['calendar']['start_date'].min()
        return datetime.datetime.strptime(s, '%Y%m%d').date()
    return idag

# ------------------------------------------------------------------ turernas första avgång
def trip_forsta_avgang(gtfs):
    """DataFrame: en rad per trip med route_id, direction_id, service_id och avgångstid (min) vid första hållplats."""
    st = gtfs['stop_times'].copy()
    st['dep_min'] = st['departure_time'].map(gtfs_tid_min)
    forsta = st.sort_values('stop_sequence').groupby('trip_id', as_index=False).first()[['trip_id', 'dep_min']]
    tr = gtfs['trips'][['trip_id', 'route_id', 'direction_id', 'service_id']]
    return tr.merge(forsta, on='trip_id', how='left')

# ------------------------------------------------------------------ linjestatistik
def linje_statistik(gtfs, datum, fm=(6, 9), em=(15, 18)):
    """Per linje: avgångar/dag, avgångar/timme i FM- resp EM-rusning, per riktning + riktningsstatus."""
    aktiva = aktiva_services(gtfs, datum)
    tf = trip_forsta_avgang(gtfs)
    tf = tf[tf['service_id'].isin(aktiva)]
    routes = gtfs['routes']
    namn = {r['route_id']: (r.get('route_short_name') or r.get('route_id'),
                            r.get('route_long_name', '')) for _, r in routes.iterrows()}
    fm_lo, fm_hi = fm[0]*60, fm[1]*60
    em_lo, em_hi = em[0]*60, em[1]*60
    fm_h = max(1, fm[1]-fm[0]); em_h = max(1, em[1]-em[0])
    stat = {}
    for rid in routes['route_id']:
        sub = tf[tf['route_id'] == rid]
        per_dir = {}
        for d in sorted(sub['direction_id'].unique()):
            s = sub[sub['direction_id'] == d]
            dep = s['dep_min'].dropna()
            per_dir[d] = {
                'avg_dag': int(len(s)),
                'fm_antal': int(((dep >= fm_lo) & (dep < fm_hi)).sum()),
                'em_antal': int(((dep >= em_lo) & (dep < em_hi)).sum()),
                'forsta': _mmhh(dep.min()) if len(dep) else '-',
                'sista': _mmhh(dep.max()) if len(dep) else '-',
            }
            per_dir[d]['fm_per_h'] = round(per_dir[d]['fm_antal']/fm_h, 1)
            per_dir[d]['em_per_h'] = round(per_dir[d]['em_antal']/em_h, 1)
        stat[rid] = {
            'kort': namn[rid][0], 'lang': namn[rid][1],
            'avg_dag': int(len(sub)),
            'riktningar': sorted(sub['direction_id'].unique()),
            'bada_riktningar': len(sub['direction_id'].unique()) >= 2,
            'per_dir': per_dir,
        }
    return stat

def _mmhh(m):
    if m is None or (isinstance(m, float) and math.isnan(m)):
        return '-'
    m = int(round(m)); return f'{(m//60)%24:02d}:{m%60:02d}'

In [ ]:
def linje_geometri(gtfs, route_id, direction):
    """Koordinatsekvens (lat,lon) för en linje+riktning – från shapes om möjligt, annars hållplatsföljden."""
    trips = gtfs['trips']
    tsub = trips[(trips['route_id'] == route_id) & (trips['direction_id'] == str(direction))]
    if not len(tsub):
        return []
    if 'shapes' in gtfs and 'shape_id' in tsub.columns and tsub.iloc[0].get('shape_id', ''):
        sid = tsub.iloc[0]['shape_id']
        s = gtfs['shapes']
        s = s[s['shape_id'] == sid].copy()
        if len(s):
            s['seq'] = pd.to_numeric(s['shape_pt_sequence'], errors='coerce')
            s = s.sort_values('seq')
            return list(zip(s['shape_pt_lat'].astype(float), s['shape_pt_lon'].astype(float)))
    # fallback: hållplatsföljd från en representativ trip
    tid = tsub.iloc[0]['trip_id']
    st = gtfs['stop_times']
    st = st[st['trip_id'] == tid].sort_values('stop_sequence')
    stops = gtfs['stops'].set_index('stop_id')
    coords = []
    for sid_ in st['stop_id']:
        if sid_ in stops.index:
            coords.append((float(stops.loc[sid_, 'stop_lat']), float(stops.loc[sid_, 'stop_lon'])))
    return coords

def _offset_linje(coords, meter):
    """Förskjuter en linje vinkelrätt `meter` meter (så båda riktningar syns bredvid varandra)."""
    if len(coords) < 2 or not meter:
        return [list(c) for c in coords]
    out = []
    for i in range(len(coords)):
        la, lo = coords[i]
        a = coords[max(0, i-1)]; b = coords[min(len(coords)-1, i+1)]
        mlat = 111320.0; mlon = 111320.0*math.cos(math.radians(la))
        vx = (b[1]-a[1])*mlon; vy = (b[0]-a[0])*mlat
        n = math.hypot(vx, vy) or 1.0
        pvx, pvy = -vy/n, vx/n                      # perpendikel (meter)
        out.append([la + (meter*pvy)/mlat, lo + (meter*pvx)/mlon])
    return out

# ------------------------------------------------------------------ interaktiv karta
FARGER = ['#e6194B', '#3cb44b', '#4363d8', '#911eb4', '#f58231', '#469990',
          '#800000', '#000075', '#9A6324', '#808000', '#f032e6', '#42d4f4']

def _popup_html(s):
    rows = ''
    for d in s.get('riktningar', []):
        pd_ = s['per_dir'][d]
        rikt = 'utåt (0)' if d == '0' else ('retur (1)' if d == '1' else f'riktn {d}')
        rows += (f"<tr><td>{rikt}</td><td style='text-align:center'>{pd_['avg_dag']}</td>"
                 f"<td style='text-align:center'>{pd_['fm_per_h']}/h</td>"
                 f"<td style='text-align:center'>{pd_['em_per_h']}/h</td>"
                 f"<td>{pd_['forsta']}–{pd_['sista']}</td></tr>")
    rikt_txt = 'Båda riktningar' if s.get('bada_riktningar') else 'En riktning'
    return (f"<div style='font-family:sans-serif'><b>Linje {s.get('kort','')}</b> – {s.get('lang','')}<br>"
            f"<b>{rikt_txt}</b> · {s.get('avg_dag',0)} avgångar/dag"
            f"<table border='1' cellpadding='3' style='border-collapse:collapse;font-size:12px;margin-top:4px'>"
            f"<tr><th>Riktning</th><th>Avg/dag</th><th>FM 6–9</th><th>EM 15–18</th><th>Första–sista</th></tr>"
            f"{rows}</table></div>")

def rita_natverk(gtfs, stat, valda=None, filnamn=None, visa_pilar=True, titel='Ålands bussnät (GTFS)'):
    """Interaktiv Folium-karta över hela nätet. Valda linjer framhävs; klick på linje visar statistik + riktning."""
    import folium
    from folium.plugins import PolyLineTextPath
    valda = set(valda or [])
    stops = gtfs['stops']
    m = folium.Map(location=[float(stops['stop_lat'].mean()), float(stops['stop_lon'].mean())],
                   zoom_start=10, tiles='OpenStreetMap', control_scale=True)
    routes = list(gtfs['routes']['route_id'])
    for idx, rid in enumerate(routes):
        s = stat.get(rid, {})
        color = FARGER[idx % len(FARGER)]
        i_urval = (rid in valda)
        opacity = 0.9 if (i_urval or not valda) else 0.25
        weight = 6 if i_urval else 3.5
        namn = f"Linje {s.get('kort', rid)}"
        fg = folium.FeatureGroup(name=namn + (' ✔' if i_urval else ''), show=True)
        for d in s.get('riktningar', ['0']):
            coords = linje_geometri(gtfs, rid, d)
            if len(coords) < 2:
                continue
            off = 14 if d == '1' else (-14 if s.get('bada_riktningar') else 0)
            coords2 = _offset_linje(coords, off)
            pl = folium.PolyLine(coords2, color=color, weight=weight, opacity=opacity,
                                 popup=folium.Popup(_popup_html(s), max_width=340), tooltip=namn)
            pl.add_to(fg)
            if visa_pilar:
                PolyLineTextPath(pl, '  ▶  ', repeat=True, offset=7,
                                 attributes={'fill': color, 'font-weight': 'bold', 'font-size': '15'}).add_to(fg)
        fg.add_to(m)
    folium.LayerControl(collapsed=True).add_to(m)
    titel_html = (f"<div style='position:fixed;top:10px;left:60px;z-index:9999;background:white;"
                  f"padding:6px 10px;border-radius:6px;font-family:sans-serif;font-size:14px;"
                  f"box-shadow:0 1px 4px rgba(0,0,0,.3)'><b>{titel}</b><br>"
                  f"<span style='font-size:11px'>▶ = färdriktning · klicka på en linje för statistik</span></div>")
    m.get_root().html.add_child(folium.Element(titel_html))
    if filnamn:
        m.save(filnamn)
    return m

def rita_natverk_statisk(gtfs, stat, valda=None, filnamn=None, titel='Ålands bussnät (GTFS)'):
    """Statisk matplotlib-karta med riktningspilar (reserv om Folium ej kan visas)."""
    import matplotlib.pyplot as plt
    valda = set(valda or [])
    fig, ax = plt.subplots(figsize=(12, 10))
    routes = list(gtfs['routes']['route_id'])
    for idx, rid in enumerate(routes):
        s = stat.get(rid, {}); color = FARGER[idx % len(FARGER)]
        vald = rid in valda
        lw = 3.5 if vald else 1.8
        alpha = 0.95 if (vald or not valda) else 0.3
        for k, d in enumerate(s.get('riktningar', ['0'])):
            coords = linje_geometri(gtfs, rid, d)
            if len(coords) < 2:
                continue
            off = 120 if d == '1' else (-120 if s.get('bada_riktningar') else 0)
            c = _offset_linje(coords, off)
            lat = [p[0] for p in c]; lon = [p[1] for p in c]
            lbl = f"Linje {s.get('kort', rid)}" + (' ✔' if vald else '') if k == 0 else None
            ax.plot(lon, lat, '-', color=color, lw=lw, alpha=alpha, label=lbl, zorder=3 if vald else 2)
            for frac in (0.3, 0.62):                       # riktningspilar
                i = int(frac*(len(c)-1)); j = min(i+1, len(c)-1)
                if j > i:
                    ax.annotate('', xy=(lon[j], lat[j]), xytext=(lon[i], lat[i]),
                                arrowprops=dict(arrowstyle='-|>', color=color, lw=lw, alpha=alpha), zorder=4)
    ax.scatter(gtfs['stops']['stop_lon'], gtfs['stops']['stop_lat'], s=6, c='#999', zorder=1)
    ax.set_aspect(1/math.cos(math.radians(float(gtfs['stops']['stop_lat'].mean()))))
    ax.set_title(titel); ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud')
    ax.legend(loc='upper right', fontsize=8, framealpha=0.9)
    plt.tight_layout()
    if filnamn:
        plt.savefig(filnamn, dpi=130)
    plt.show()

## Steg 1 – Välj GTFS-data
Sätt `GTFS_PATH` till mappen eller `.zip`-filen med dina GTFS-filer (`stops.txt`, `routes.txt`, `trips.txt`,
`stop_times.txt`, …). Lämnar du den tom används det **medföljande Åland-exemplet**, och i Colab erbjuds
uppladdning av en zip.

In [ ]:
GTFS_PATH = ''   # <-- ange din GTFS-mapp eller .zip här (t.ex. '/content/drive/MyDrive/aland_gtfs' eller 'gtfs.zip')

# Reserv: medföljande exempel, annars uppladdning i Colab
if not GTFS_PATH:
    for kand in ['data/gtfs_sample', '../data/gtfs_sample', 'gtfs_sample', 'gtfs', 'gtfs.zip']:
        if os.path.exists(kand):
            GTFS_PATH = kand
            print(f'Ingen sökväg angiven – använder {"medföljande exempel" if "sample" in kand else kand}: {kand}')
            break
if not GTFS_PATH and 'google.colab' in sys.modules:
    from google.colab import files
    print('Ladda upp din GTFS-zip:')
    GTFS_PATH = list(files.upload().keys())[0]
assert GTFS_PATH, 'Sätt GTFS_PATH till din GTFS-mapp eller .zip och kör om cellen.'

gtfs = las_gtfs(GTFS_PATH)
print('Inläst:', {k: len(v) for k, v in gtfs.items()})

# utdatamapp
UT = next((p for p in ['output', '../output'] if os.path.isdir(p)), '.')

## Steg 2 – Trafikdygn och statistik
Statistiken beräknas för ett **vardagsdygn** (default väljs automatiskt). Rusningsfönstren är
förmiddag **06–09** och eftermiddag **15–18** (ändra vid behov).

In [ ]:
DATUM      = ''          # '' => välj automatiskt ett vardagsdatum med trafik; annars 'YYYY-MM-DD'
FM_RUSNING = (6, 9)      # förmiddagsrusning
EM_RUSNING = (15, 18)    # eftermiddagsrusning

datum = datetime.date.fromisoformat(DATUM) if DATUM else valj_standarddatum(gtfs)
veckodag = ['måndag','tisdag','onsdag','torsdag','fredag','lördag','söndag'][datum.weekday()]
stat = linje_statistik(gtfs, datum, fm=FM_RUSNING, em=EM_RUSNING)
print(f'Trafikdygn: {datum} ({veckodag})  ·  {len(stat)} linjer')

### Översiktstabell över linjerna

In [ ]:
rader = []
for rid in gtfs['routes']['route_id']:
    s = stat[rid]
    fm = max((s['per_dir'][d]['fm_per_h'] for d in s['per_dir']), default=0)
    em = max((s['per_dir'][d]['em_per_h'] for d in s['per_dir']), default=0)
    rader.append({'route_id': rid, 'linje': s['kort'], 'namn': s['lang'],
                  'avg/dag': s['avg_dag'], 'riktning': 'båda' if s['bada_riktningar'] else 'en',
                  'FM/h (max)': fm, 'EM/h (max)': em})
linjetabell = pd.DataFrame(rader)
linjetabell

## Steg 3 – Välj linjer att arbeta vidare med
Markera de linjer du vill avgränsa till (Ctrl/Cmd-klick för flera). Kör sedan kartcellen nedan – valda linjer
framhävs. Saknas `ipywidgets` kan du i stället sätta `VALDA_LINJER` manuellt, t.ex. `VALDA_LINJER = ['1', '4']`.

In [ ]:
VALDA_LINJER = []   # sätt manuellt om du vill, t.ex. ['1', '4']
linje_val = None
try:
    import ipywidgets as widgets
    from IPython.display import display
    opts = [(f"Linje {stat[r]['kort']} – {stat[r]['lang']}", r) for r in gtfs['routes']['route_id']]
    linje_val = widgets.SelectMultiple(options=opts, description='Linjer:',
                                       rows=min(12, len(opts)), layout=widgets.Layout(width='70%'))
    display(linje_val)
    print('Markera linjer och kör nästa cell (kartan).')
except Exception:
    print('ipywidgets saknas – sätt VALDA_LINJER manuellt ovan, t.ex. ["1","4"].')

## Steg 4 – Karta över hela nätet
Hela nätet ritas; **valda linjer framhävs**. Pilar (▶) visar färdriktning – linjer som går åt båda hållen får två
parallella spår, enkelriktade bara ett. **Klicka på en linje** för avgångar/dag, FM/EM-rusning och riktning.

In [ ]:
valda = list(linje_val.value) if (linje_val is not None and linje_val.value) else VALDA_LINJER
print('Valda linjer:', valda if valda else '(inga valda – hela nätet visas)')

if HAR_FOLIUM:
    karta = rita_natverk(gtfs, stat, valda=valda, filnamn=os.path.join(UT, 'gtfs_natverk.html'))
    print('Interaktiv karta sparad:', os.path.join(UT, 'gtfs_natverk.html'))
else:
    rita_natverk_statisk(gtfs, stat, valda=valda, filnamn=os.path.join(UT, 'gtfs_natverk.png'))
    karta = None
karta

## Steg 5 – Spara urvalet
De valda linjerna sparas för det fortsatta arbetet (t.ex. framtida optimering).

In [ ]:
urval = valda if valda else list(gtfs['routes']['route_id'])
urval_df = pd.DataFrame({'route_id': urval,
                         'linje': [stat[r]['kort'] for r in urval],
                         'namn': [stat[r]['lang'] for r in urval]})
urval_df.to_csv(os.path.join(UT, 'valda_linjer.csv'), index=False)
print('Sparade urval till valda_linjer.csv:')
urval_df